# ClusterFlow validation — *Klebsiella pneumoniae* neonatal outbreak

This notebook runs ClusterFlow end-to-end on the neonatal-unit *K. pneumoniae* outbreak (Kwenda et al. 2024) and compares the result against SNP2Cluster v0.5.4.

**Acceptance gates** (from `transmission_cluster_tool_plan.md` §8):
1. ARI between ClusterFlow consensus and SNP2Cluster ≥ 0.85 on the four main clusters.
2. ClusterFlow detects ≥ 2 additional clusters that SNP2Cluster missed (specifically ST25 and ST39).
3. Pipeline completes in < 60 s on a 4-core laptop.
4. Two runs with the same seed produce bit-identical assignments.

If the Kwenda dataset has not been placed at `tests/fixtures/kp_real/`, the notebook automatically falls back to the synthetic 28-isolate fixture so it still runs reproducibly.

In [ ]:
from pathlib import Path
import json
import logging
import time

import pandas as pd
from IPython.display import Image, display
from sklearn.metrics import adjusted_rand_score

from clusterflow.config import ClusterFlowConfig
from clusterflow.pipeline import run_pipeline

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s', datefmt='%H:%M:%S')

REPO = Path('..').resolve()
REAL = REPO / 'tests' / 'fixtures' / 'kp_real'
FALLBACK_CONFIG = REPO / 'tests' / 'fixtures' / 'kp_config.yaml'

if REAL.exists() and (REAL / 'kp_config.yaml').exists():
    cfg_path = REAL / 'kp_config.yaml'
    print(f'using REAL Kwenda dataset: {cfg_path}')
else:
    cfg_path = FALLBACK_CONFIG
    print(f'real dataset not present — using synthetic fixture: {cfg_path}')
    print('To run on real data: place the Kwenda et al. files in tests/fixtures/kp_real/')

cfg = ClusterFlowConfig.from_yaml(cfg_path)
out_dir = REPO / 'notebooks' / '_validation_results'
cfg = cfg.model_copy(update={'output_dir': out_dir})
out_dir.mkdir(parents=True, exist_ok=True)
print(cfg.model_dump_json(indent=2))

## 1. Run the pipeline (timed)

In [ ]:
t0 = time.perf_counter()
result = run_pipeline(cfg)
elapsed = time.perf_counter() - t0
print(f'pipeline completed in {elapsed:.2f} s — {result.consensus.n_clusters} clusters across {result.n_isolates} isolates')
assert elapsed < 60, 'failed performance gate (>60 s)'

## 2. Per-cluster summary

In [ ]:
rows = [
    {
        'cluster_id': c.cluster_id,
        'n_isolates': len(c.isolate_ids),
        'sequence_types': ', '.join(c.sequence_types) or '-',
        'wards': ', '.join(c.wards),
        'date_range': f'{c.date_range[0]} → {c.date_range[1]}',
        'index_case': c.index_case_candidate,
        'confidence': round(c.confidence, 3),
    }
    for c in result.transmission_clusters
]
pd.DataFrame(rows)

## 3. Per-method cluster counts and consensus agreement

In [ ]:
method_counts = {m: a.n_clusters for m, a in result.cluster_assignments.items()}
method_counts['consensus'] = result.consensus.n_clusters
print(json.dumps(method_counts, indent=2))
agree = pd.Series(result.consensus.agreement_score).describe()
agree.name = 'agreement_score'
agree.to_frame()

## 4. Reproducibility gate

In [ ]:
result_b = run_pipeline(cfg, run_viz=False)
assert result.consensus.assignments == result_b.consensus.assignments, 'reproducibility gate failed'
print('reproducibility gate passed — same seed → identical assignments')

## 5. Compare to SNP2Cluster baseline (if available)

If the SNP2Cluster CSV (one row per isolate with column `cluster`) is at `tests/fixtures/kp_real/snp2cluster_assignments.csv`, we compute Adjusted Rand Index and check that ClusterFlow detects extra clusters.

In [ ]:
snp2c_path = REAL / 'snp2cluster_assignments.csv'
if snp2c_path.exists():
    snp2c = pd.read_csv(snp2c_path).set_index('isolate_id')['cluster'].to_dict()
    common = sorted(set(snp2c) & set(result.consensus.assignments))
    truth = [snp2c[i] for i in common]
    pred = [result.consensus.assignments[i] for i in common]
    ari = adjusted_rand_score(truth, pred)
    extra = result.consensus.n_clusters - len(set(truth))
    print(f'ARI vs SNP2Cluster: {ari:.3f}')
    print(f'extra clusters detected by ClusterFlow: {extra}')
    assert ari >= 0.85, 'ARI gate failed (<0.85)'
    assert extra >= 2, 'extra-cluster detection gate failed (<2)'
else:
    print(f'no SNP2Cluster baseline at {snp2c_path} — skipping ARI gate')

## 6. Embedded figures (Phase 6.1)

In [ ]:
for fig_name in ['snp_heatmap', 'minimum_spanning_tree', 'epi_timeline_scatter', 'cluster_comparison_grid', 'bootstrap_stability_terrain']:
    p = out_dir / 'figures' / f'{fig_name}.png'
    if p.exists():
        print(fig_name)
        display(Image(filename=str(p)))